## prediction time:

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1' 

In [4]:
import os
import pandas as pd
import numpy as np
import torch
import joblib
import xgboost as xgb
from features import (
    build_team_match_index, build_goals_index, build_shootout_index,
    calculate_elo_history, build_match_features, build_score_matrix,
    FEATURE_COLS
)

os.makedirs('../predictions', exist_ok=True)

# --- Load data + rebuild indices from full history (as of today) ---
clean_df = pd.read_csv('../data/processed/clean_matches.csv', parse_dates=['date'])
goals_df = pd.read_csv('../data/raw/goalscorers.csv', parse_dates=['date'])
shootouts_df = pd.read_csv('../data/raw/shootouts.csv', parse_dates=['date'])
upcoming = pd.read_csv('../data/wc2026/upcoming_fixtures.csv', parse_dates=['date'])

team_index = build_team_match_index(clean_df)
goals_index = build_goals_index(goals_df)
shootout_index = build_shootout_index(shootouts_df)
elo_history, _ = calculate_elo_history(clean_df)

# --- Load trained models ---
xgb_model = xgb.XGBClassifier()
xgb_model.load_model('../models/xgboost_final_model.json')

class LambdaNet(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 64), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(64, 32), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(32, 2), torch.nn.Softplus()
        )
    def forward(self, x):
        return torch.clamp(self.net(x), min=0.2, max=5.0)

lambda_model = LambdaNet(input_dim=len(FEATURE_COLS))
lambda_model.load_state_dict(torch.load('../models/best_lambda_model.pth'))
lambda_model.eval()
scaler = joblib.load('../models/scaler.pkl')  # save this now if not already: joblib.dump(scaler, '../models/scaler.pkl')

RESULT_LABELS = ['Away Win', 'Draw', 'Home Win']

def predict_match(home_team, away_team, match_date):
    feat_dict = build_match_features(home_team, away_team, match_date, team_index, goals_index, shootout_index, elo_history)
    X_row = pd.DataFrame([feat_dict])[FEATURE_COLS]

    # XGBoost -- Win/Draw/Loss
    wdl_proba = xgb_model.predict_proba(X_row)[0]  # [away, draw, home]

    # LambdaNet -- expected goals
    X_scaled = scaler.transform(X_row.values)
    with torch.no_grad():
        lambdas = lambda_model(torch.tensor(X_scaled, dtype=torch.float32)).numpy()[0]
    lambda_home, lambda_away = lambdas[0], lambdas[1]

    # Poisson scoreline matrix, built from LambdaNet's lambdas
    score_matrix = build_score_matrix(lambda_home, lambda_away, max_goals=6)
    most_likely_idx = np.unravel_index(score_matrix.argmax(), score_matrix.shape)

    flat = [(i, j, score_matrix[i,j]) for i in range(7) for j in range(7)]
    top5 = sorted(flat, key=lambda x: x[2], reverse=True)[:5]

    return {
        'home_team': home_team, 'away_team': away_team, 'date': match_date,
        'xgb_away_win': wdl_proba[0], 'xgb_draw': wdl_proba[1], 'xgb_home_win': wdl_proba[2],
        'predicted_outcome': RESULT_LABELS[np.argmax(wdl_proba)],
        'lambda_home': lambda_home, 'lambda_away': lambda_away,
        'most_likely_score': f"{most_likely_idx[0]}-{most_likely_idx[1]}",
        'top5_scorelines': [(f"{i}-{j}", round(p*100,1)) for i,j,p in top5],
    }

# --- Run on all remaining real fixtures ---
predictions = []
for _, row in upcoming.iterrows():
    pred = predict_match(row['home_team'], row['away_team'], row['date'])
    predictions.append(pred)
    print(f"\n{'='*55}\n{pred['home_team']} vs {pred['away_team']} ({pred['date'].date()})")
    print(f"  Win/Draw/Loss : Home {pred['xgb_home_win']*100:.1f}% | Draw {pred['xgb_draw']*100:.1f}% | Away {pred['xgb_away_win']*100:.1f}%")
    print(f"  Predicted outcome : {pred['predicted_outcome']}")
    print(f"  λ_home={pred['lambda_home']:.2f}, λ_away={pred['lambda_away']:.2f}")
    print(f"  Most likely score : {pred['most_likely_score']}")
    print(f"  Top 5 scorelines  : {pred['top5_scorelines']}")

pd.DataFrame(predictions).to_csv('../predictions/predictions_log.csv', index=False)
print(f"\n\n✅ Saved {len(predictions)} predictions to predictions_log.csv")


France vs Morocco (2026-07-09)
  Win/Draw/Loss : Home 34.3% | Draw 38.7% | Away 27.0%
  Predicted outcome : Draw
  λ_home=1.37, λ_away=0.99
  Most likely score : 1-0
  Top 5 scorelines  : [('1-0', np.float64(13.0)), ('1-1', np.float64(12.8)), ('0-0', np.float64(9.5)), ('0-1', np.float64(9.3)), ('2-0', np.float64(8.9))]

Spain vs Belgium (2026-07-10)
  Win/Draw/Loss : Home 55.5% | Draw 33.5% | Away 10.9%
  Predicted outcome : Home Win
  λ_home=2.04, λ_away=0.79
  Most likely score : 2-0
  Top 5 scorelines  : [('2-0', np.float64(12.3)), ('1-0', np.float64(12.1)), ('2-1', np.float64(9.8)), ('1-1', np.float64(9.6)), ('3-0', np.float64(8.4))]

Norway vs England (2026-07-11)
  Win/Draw/Loss : Home 10.4% | Draw 36.2% | Away 53.3%
  Predicted outcome : Away Win
  λ_home=0.87, λ_away=1.62
  Most likely score : 0-1
  Top 5 scorelines  : [('0-1', np.float64(13.4)), ('1-1', np.float64(11.7)), ('0-2', np.float64(10.9)), ('1-2', np.float64(9.5)), ('0-0', np.float64(8.2))]

Argentina vs Switzerland 

In [6]:
# --- Semifinals, based on your confirmed quarterfinal winners ---
semifinals = [
    {"home_team": "France", "away_team": "Spain", "date": pd.Timestamp("2026-07-14")},
    {"home_team": "England", "away_team": "Argentina", "date": pd.Timestamp("2026-07-15")},
]

sf_predictions = []
for match in semifinals:
    pred = predict_match(match['home_team'], match['away_team'], match['date'])
    sf_predictions.append(pred)
    print(f"\n{'='*55}\n{pred['home_team']} vs {pred['away_team']} ({pred['date'].date()})")
    print(f"  Win/Draw/Loss : Home {pred['xgb_home_win']*100:.1f}% | Draw {pred['xgb_draw']*100:.1f}% | Away {pred['xgb_away_win']*100:.1f}%")
    print(f"  Predicted outcome : {pred['predicted_outcome']}")
    print(f"  λ_home={pred['lambda_home']:.2f}, λ_away={pred['lambda_away']:.2f}")
    print(f"  Most likely score : {pred['most_likely_score']}")
    print(f"  Top 5 scorelines  : {pred['top5_scorelines']}")

pd.DataFrame(sf_predictions).to_csv('../predictions/semifinal_predictions_log.csv', index=False)
print(f"\n\n✅ Saved {len(sf_predictions)} semifinal predictions")


France vs Spain (2026-07-14)
  Win/Draw/Loss : Home 30.6% | Draw 38.1% | Away 31.3%
  Predicted outcome : Draw
  λ_home=1.06, λ_away=1.06
  Most likely score : 1-1
  Top 5 scorelines  : [('1-1', np.float64(13.5)), ('0-1', np.float64(12.7)), ('1-0', np.float64(12.7)), ('0-0', np.float64(11.9)), ('1-2', np.float64(7.2))]

England vs Argentina (2026-07-15)
  Win/Draw/Loss : Home 25.6% | Draw 37.7% | Away 36.7%
  Predicted outcome : Draw
  λ_home=1.38, λ_away=1.17
  Most likely score : 1-1
  Top 5 scorelines  : [('1-1', np.float64(12.6)), ('1-0', np.float64(10.8)), ('0-1', np.float64(9.2)), ('2-1', np.float64(8.7)), ('0-0', np.float64(7.8))]


✅ Saved 2 semifinal predictions


In [8]:
from features import (
    build_team_match_index, build_goals_index, build_shootout_index,
    calculate_elo_history, build_match_features, build_score_matrix,
    get_shootout_win_rate   # ← add this
)

In [9]:
def resolve_knockout_prediction(pred):
    """
    Takes the dict from predict_match(). A knockout match can't legitimately end
    in a draw, so if Draw is the top predicted class, this reallocates that
    probability between both teams using their shootout win rate (the same
    feature used elsewhere in the model) to get a genuine advancement probability.
    """
    home_sw = get_shootout_win_rate(pred['home_team'], pred['date'], shootout_index)
    away_sw = get_shootout_win_rate(pred['away_team'], pred['date'], shootout_index)
    total_sw = home_sw + away_sw

    p_home_adv = pred['xgb_home_win'] + pred['xgb_draw'] * (home_sw / total_sw)
    p_away_adv = pred['xgb_away_win'] + pred['xgb_draw'] * (away_sw / total_sw)

    pred['home_shootout_rate'] = home_sw
    pred['away_shootout_rate'] = away_sw
    pred['p_home_advances'] = p_home_adv
    pred['p_away_advances'] = p_away_adv
    pred['projected_advancer'] = pred['home_team'] if p_home_adv > p_away_adv else pred['away_team']
    pred['draw_resolved'] = pred['predicted_outcome'] == 'Draw'

    if pred['draw_resolved']:
        print(f"  ⚠ Draw predicted — not a valid knockout result. Resolving via shootout rate.")
        print(f"     Shootout win rate  : {pred['home_team']} {home_sw:.4f}  |  {pred['away_team']} {away_sw:.4f}")
        print(f"     P({pred['home_team']} advances) : {p_home_adv*100:.1f}%")
        print(f"     P({pred['away_team']} advances) : {p_away_adv*100:.1f}%")
    else:
        print(f"  Outright result predicted — no tiebreak needed.")
    print(f"     Projected advancer : {pred['projected_advancer']}")

    return pred


# --- Run on a list of knockout matches, resolving draws, printing, and logging ---
import os

def predict_knockout_round(matches, log_path='../predictions/knockout_predictions_log.csv'):
    round_predictions = []
    for home_team, away_team, date in matches:
        pred = predict_match(home_team, away_team, pd.Timestamp(date))
        pred = resolve_knockout_prediction(pred)
        round_predictions.append(pred)

        print(f"\n{'='*55}\n{pred['home_team']} vs {pred['away_team']} ({pred['date'].date()})")
        print(f"  Win/Draw/Loss : Home {pred['xgb_home_win']*100:.1f}% | Draw {pred['xgb_draw']*100:.1f}% | Away {pred['xgb_away_win']*100:.1f}%")
        print(f"  λ_home={pred['lambda_home']:.2f}, λ_away={pred['lambda_away']:.2f}")
        print(f"  Most likely score : {pred['most_likely_score']}")
        print(f"  Top 5 scorelines  : {pred['top5_scorelines']}")

    df = pd.DataFrame(round_predictions)
    if os.path.exists(log_path):
        df.to_csv(log_path, mode='a', header=False, index=False)
    else:
        df.to_csv(log_path, index=False)
    print(f"\n\n✅ Logged {len(round_predictions)} predictions to {log_path}")
    return round_predictions


# Example: semifinals
semifinals = [
    ("France", "Spain", "2026-07-14"),
    ("England", "Argentina", "2026-07-15"),
]
sf_predictions = predict_knockout_round(semifinals)

  ⚠ Draw predicted — not a valid knockout result. Resolving via shootout rate.
     Shootout win rate  : France 0.4643  |  Spain 0.5000
     P(France advances) : 48.9%
     P(Spain advances) : 51.1%
     Projected advancer : Spain

France vs Spain (2026-07-14)
  Win/Draw/Loss : Home 30.6% | Draw 38.1% | Away 31.3%
  λ_home=1.06, λ_away=1.06
  Most likely score : 1-1
  Top 5 scorelines  : [('1-1', np.float64(13.5)), ('0-1', np.float64(12.7)), ('1-0', np.float64(12.7)), ('0-0', np.float64(11.9)), ('1-2', np.float64(7.2))]
  ⚠ Draw predicted — not a valid knockout result. Resolving via shootout rate.
     Shootout win rate  : England 0.3667  |  Argentina 0.6346
     P(England advances) : 39.4%
     P(Argentina advances) : 60.6%
     Projected advancer : Argentina

England vs Argentina (2026-07-15)
  Win/Draw/Loss : Home 25.6% | Draw 37.7% | Away 36.7%
  λ_home=1.38, λ_away=1.17
  Most likely score : 1-1
  Top 5 scorelines  : [('1-1', np.float64(12.6)), ('1-0', np.float64(10.8)), ('0-1', n

## final prediction:

In [10]:
final_and_third = [
    ("France", "England", "2026-07-18"),      # 3rd Place — SF losers — Miami
    ("Spain", "Argentina", "2026-07-19"),      # Final — SF winners — MetLife Stadium
]

final_predictions = predict_knockout_round(final_and_third)

  Outright result predicted — no tiebreak needed.
     Projected advancer : France

France vs England (2026-07-18)
  Win/Draw/Loss : Home 37.0% | Draw 36.7% | Away 26.2%
  λ_home=1.22, λ_away=1.09
  Most likely score : 1-1
  Top 5 scorelines  : [('1-1', np.float64(13.2)), ('1-0', np.float64(12.1)), ('0-1', np.float64(10.8)), ('0-0', np.float64(9.9)), ('2-1', np.float64(8.1))]
  ⚠ Draw predicted — not a valid knockout result. Resolving via shootout rate.
     Shootout win rate  : Spain 0.5000  |  Argentina 0.6346
     P(Spain advances) : 51.2%
     P(Argentina advances) : 48.8%
     Projected advancer : Spain

Spain vs Argentina (2026-07-19)
  Win/Draw/Loss : Home 33.9% | Draw 39.4% | Away 26.7%
  λ_home=1.52, λ_away=1.01
  Most likely score : 1-1
  Top 5 scorelines  : [('1-1', np.float64(12.2)), ('1-0', np.float64(12.1)), ('2-1', np.float64(9.3)), ('2-0', np.float64(9.2)), ('0-1', np.float64(8.0))]


✅ Logged 2 predictions to ../predictions/knockout_predictions_log.csv
